In [1]:
!git clone https://github.com/VinAIResearch/COVID19Tweet.git
%cd COVID19Tweet
!unzip -q WNUT-2020-Task-2-Dataset.zip -d data
!find data -type f

fatal: destination path 'COVID19Tweet' already exists and is not an empty directory.
/content/COVID19Tweet
replace data/WNUT-2020-Task-2-Dataset/train.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace data/WNUT-2020-Task-2-Dataset/test.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace data/WNUT-2020-Task-2-Dataset/unlabeled_test_with_noise.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace data/WNUT-2020-Task-2-Dataset/valid.tsv? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
data/WNUT-2020-Task-2-Dataset/train.tsv
data/WNUT-2020-Task-2-Dataset/test.tsv
data/WNUT-2020-Task-2-Dataset/unlabeled_test_with_noise.tsv
data/WNUT-2020-Task-2-Dataset/valid.tsv


In [2]:
# 1. 함수 가져오기
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer #문자를 숫자로 가져옴
from sklearn.linear_model import LogisticRegression #로지스틱회귀 사용
from sklearn.metrics import accuracy_score, f1_score, classification_report #정확도, 점수, 성능표 보기

In [3]:
# 2. 데이터 읽기
train_df = pd.read_csv("data/WNUT-2020-Task-2-Dataset/train.tsv", sep="\t") #탭으로 나눠져있음
valid_df = pd.read_csv("data/WNUT-2020-Task-2-Dataset/valid.tsv", sep="\t", header=None)
valid_df.columns = ['Id', 'Text', 'Label'] #헤더가 없다고 함 + 열 이름 할당하게 되어 이름도 다시 할당

In [4]:
# 3. 데이터 확인
train_df.head()
train_df.columns
train_df.shape

(6936, 3)

In [5]:
# 4. 입력과 정답 나누기

X_train = train_df["Text"].astype(str)
y_train = train_df["Label"].astype(str)

X_valid = valid_df["Text"].astype(str)
y_valid = valid_df["Label"].astype(str)

In [6]:
# 5. 텍스트 숫자화
vectorizer = TfidfVectorizer()

X_train_vec = vectorizer.fit_transform(X_train) #단어사전 만든 후 숫자로 바꿈
X_valid_vec = vectorizer.transform(X_valid) #train에서 만든 기준으로 숫자로만 바꾼다

In [7]:
# 6. 모델 학습
model = LogisticRegression(max_iter=1000) #로지스틱 회귀 모델 만들기
model.fit(X_train_vec, y_train) #학습 시작

LogisticRegression(max_iter=1000)

In [8]:
# 7. 예측
pred = model.predict(X_valid_vec)

In [9]:
# 8. 성능 확인
print(accuracy_score(y_valid, pred))
print(f1_score(y_valid, pred, pos_label="INFORMATIVE"))
print(classification_report(y_valid, pred))

0.817
0.7941507311586051
               precision    recall  f1-score   support

  INFORMATIVE       0.85      0.75      0.79       472
UNINFORMATIVE       0.80      0.88      0.84       528

     accuracy                           0.82      1000
    macro avg       0.82      0.81      0.81      1000
 weighted avg       0.82      0.82      0.82      1000



In [10]:
# 9. 키워드 기반 위험 점수 설정
import re
import numpy as np
import pandas as pd

def clean_for_keyword(text):
    text = str(text).lower()

    text = text.replace("httpurl", " url ")
    text = text.replace("@user", " user ")

    text = re.sub(r"http\S+|www\S+", " url ", text)
    text = re.sub(r"@\w+", " user ", text)
    text = text.replace("#", "")

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text
# 범주별로 키워드 설정
risk_keyword_groups = {
    "case_growth": [
        "new cases", "confirmed cases", "confirmed", "positive cases",
        "tested positive", "cases", "case increase", "increase",
        "rising", "rise", "surge", "spike"
    ],

    "death_severity": [
        "death", "deaths", "dead", "fatal", "fatality", "fatalities"
    ],

    "medical_system": [
        "hospital", "hospitals", "icu", "ventilator", "ventilators",
        "hospital beds", "icu beds", "beds", "capacity", "overwhelmed"
    ],

    "testing_quarantine": [
        "test", "tests", "tested", "testing", "quarantine",
        "isolation", "isolated", "self isolate", "self isolation"
    ],

    "symptoms_infection": [
        "symptom", "symptoms", "infected", "infection", "outbreak",
        "spread", "spreading", "transmission", "exposed"
    ],

    "public_response": [
        "lockdown", "shutdown", "restriction", "restrictions",
        "emergency", "curfew", "ban", "closed", "closure"
    ]
}
# 초기 가중치
risk_group_weights = {
    "case_growth": 2,
    "death_severity": 3,
    "medical_system": 3,
    "testing_quarantine": 1,
    "symptoms_infection": 1,
    "public_response": 1
}

In [11]:
# 10. 키워드 매칭 함수
def contains_keyword(cleaned_text, keyword):
    keyword = keyword.lower()

    # 여러 단어로 된 phrase keyword
    if " " in keyword:
        return keyword in cleaned_text

    # 단일 단어 keyword
    tokens = cleaned_text.split()
    return keyword in tokens

def keyword_risk_score(text):
    cleaned = clean_for_keyword(text)

    total_score = 0
    matched = {}

    for group, keywords in risk_keyword_groups.items():
        group_matches = []

        for keyword in keywords:
            if contains_keyword(cleaned, keyword):
                group_matches.append(keyword)

        if group_matches:
            weight = risk_group_weights[group]
            total_score += weight
            matched[group] = group_matches

    return total_score, matched

In [12]:
# 11. Logistic Regression 예측 결과 + 키워드 점수 결합
def predict_info_with_risk(text):
    # 1. TF-IDF 변환
    text_vec = vectorizer.transform([str(text)])

    # 2. info/non-info 예측
    pred_label = model.predict(text_vec)[0]

    # 3. INFORMATIVE 확률 구하기
    proba = model.predict_proba(text_vec)[0]
    class_to_prob = dict(zip(model.classes_, proba))

    prob_informative = class_to_prob.get("INFORMATIVE", 0.0)
    prob_uninformative = class_to_prob.get("UNINFORMATIVE", 0.0)

    # 4. 키워드 위험 점수 계산
    keyword_score, matched_keywords = keyword_risk_score(text)

    # 5. 최종 위험 점수 계산
    final_score = 0

    if pred_label == "INFORMATIVE":
        final_score += 3

    if prob_informative >= 0.8:
        final_score += 2
    elif prob_informative >= 0.6:
        final_score += 1

    final_score += keyword_score

    # 6. 안전장치: 모델이 UNINFORMATIVE라고 강하게 판단하면 HIGH 방지
    if pred_label == "UNINFORMATIVE" and prob_informative < 0.5:
        final_score = min(final_score, 3)

    # 7. LOW / MEDIUM / HIGH 구분
    if final_score >= 9 and prob_informative >= 0.8:
        alert_level = "HIGH"
    elif final_score >= 5:
        alert_level = "MEDIUM"
    else:
        alert_level = "LOW"

    return {
        "Text": text,
        "pred_label": pred_label,
        "prob_informative": prob_informative,
        "prob_uninformative": prob_uninformative,
        "keyword_score": keyword_score,
        "matched_keywords": matched_keywords,
        "final_score": final_score,
        "alert_level": alert_level
    }

In [13]:
# 12. 예시 트윗으로 테스트
demo_texts = [
    "There are 500 new confirmed COVID-19 cases and 12 deaths in the city today.",
    "The local hospital is overwhelmed and ICU beds are almost full.",
    "I hate staying home because of coronavirus.",
    "New testing centers will open tomorrow for people with symptoms.",
    "Watching movies during quarantine is boring.",
    "Officials reported a sudden spike in infections after a large outbreak."
]

for text in demo_texts:
    result = predict_info_with_risk(text)
    print("=" * 100)
    for k, v in result.items():
        print(k, ":", v)

Text : There are 500 new confirmed COVID-19 cases and 12 deaths in the city today.
pred_label : INFORMATIVE
prob_informative : 0.9920083112405145
prob_uninformative : 0.00799168875948555
keyword_score : 5
matched_keywords : {'case_growth': ['confirmed', 'cases'], 'death_severity': ['deaths']}
final_score : 10
alert_level : HIGH
Text : The local hospital is overwhelmed and ICU beds are almost full.
pred_label : UNINFORMATIVE
prob_informative : 0.2191753437349958
prob_uninformative : 0.7808246562650042
keyword_score : 3
matched_keywords : {'medical_system': ['hospital', 'icu', 'icu beds', 'beds', 'overwhelmed']}
final_score : 3
alert_level : LOW
Text : I hate staying home because of coronavirus.
pred_label : UNINFORMATIVE
prob_informative : 0.19430828104413855
prob_uninformative : 0.8056917189558614
keyword_score : 0
matched_keywords : {}
final_score : 0
alert_level : LOW
Text : New testing centers will open tomorrow for people with symptoms.
pred_label : UNINFORMATIVE
prob_informative :

In [14]:
# 13. valid  데이터 전체에 후처리 작용
valid_results = []

for text in X_valid:
    valid_results.append(predict_info_with_risk(text))

valid_risk_df = pd.DataFrame(valid_results)

valid_risk_df.head()
# 위험 단계 분포 확인
valid_risk_df["alert_level"].value_counts()
# HIGH로 잡힌 트윗 확인
valid_risk_df[valid_risk_df["alert_level"] == "HIGH"][
    ["Text", "pred_label", "prob_informative", "keyword_score", "matched_keywords", "final_score", "alert_level"]
].head(20)
# MEDIUM으로 잡힌 트윗 확인
valid_risk_df[valid_risk_df["alert_level"] == "MEDIUM"][
    ["Text", "pred_label", "prob_informative", "keyword_score", "matched_keywords", "final_score", "alert_level"]
].head(20)

,Text,pred_label,prob_informative,keyword_score,matched_keywords,final_score,alert_level
1,Second case DR 🇩🇴 The Canadian woman has not b...,INFORMATIVE,0.886427,0,{},5,MEDIUM
4,Report suggested that the actual number of und...,INFORMATIVE,0.683655,2,"{'case_growth': ['positive cases', 'cases']}",6,MEDIUM
9,UPDATE: the two people at our Hail Creek opera...,INFORMATIVE,0.753189,1,"{'testing_quarantine': ['test', 'tested', 'iso...",5,MEDIUM
16,IN PICTURES: NOA Mobile Public Address Van swi...,INFORMATIVE,0.544316,2,{'case_growth': ['increase']},5,MEDIUM
17,"The Taslee Palm City Estate in Maitama, Abuja,...",INFORMATIVE,0.688925,3,"{'case_growth': ['confirmed'], 'symptoms_infec...",7,MEDIUM
21,There are eight new cases of #COVID19 in #Sask...,INFORMATIVE,0.889007,3,"{'case_growth': ['new cases', 'cases'], 'testi...",8,MEDIUM
22,"240,000 Americans died from coronavirus and, w...",INFORMATIVE,0.515683,3,{'death_severity': ['deaths']},6,MEDIUM
24,New York statewide coronavirus total hits 22 a...,INFORMATIVE,0.900425,2,{'case_growth': ['cases']},7,MEDIUM
25,NEWS: A permanent memorial is going to be set ...,INFORMATIVE,0.627939,3,{'medical_system': ['hospital']},7,MEDIUM
26,#KatieMiller is #StephenMillers new wife &amp;...,INFORMATIVE,0.826590,0,{},5,MEDIUM


In [15]:
# 14. 키워드가 적절한지 확인
def has_group_keyword(text, keywords):
    cleaned = clean_for_keyword(text)
    return any(contains_keyword(cleaned, keyword) for keyword in keywords)

rows = []

for group, keywords in risk_keyword_groups.items():
    hit = train_df["Text"].apply(lambda x: has_group_keyword(x, keywords))

    total_hit = hit.sum()
    informative_hit = ((train_df["Label"] == "INFORMATIVE") & hit).sum()
    uninformative_hit = ((train_df["Label"] == "UNINFORMATIVE") & hit).sum()

    informative_ratio = informative_hit / total_hit if total_hit > 0 else 0

    rows.append({
        "group": group,
        "total_hit": total_hit,
        "informative_hit": informative_hit,
        "uninformative_hit": uninformative_hit,
        "informative_ratio": informative_ratio
    })

keyword_eval_df = pd.DataFrame(rows)
keyword_eval_df.sort_values("informative_ratio", ascending=False)

,group,total_hit,informative_hit,uninformative_hit,informative_ratio
0,case_growth,2502,1712,790,0.684253
3,testing_quarantine,1462,986,476,0.674419
1,death_severity,1408,923,485,0.655540
2,medical_system,608,342,266,0.562500
4,symptoms_infection,1091,485,606,0.444546
5,public_response,404,153,251,0.378713


In [16]:
# 15. 라벨 없는 데이터에 적용하고 CSV 저장
unlabeled_df = pd.read_csv(
    "data/WNUT-2020-Task-2-Dataset/unlabeled_test_with_noise.tsv",
    sep="\t",
    header=None
)

unlabeled_df.columns = ["Id", "Text"]

unlabeled_df.head()

unlabeled_results = []

for_texts = unlabeled_df["Text"].astype(str)

for text in for_texts:
    unlabeled_results.append(predict_info_with_risk(text))

unlabeled_risk_df = pd.DataFrame(unlabeled_results)

unlabeled_risk_df.head()

,Text,pred_label,prob_informative,prob_uninformative,keyword_score,matched_keywords,final_score,alert_level
0,Fox Business' Lou Dobbs Self-Quarantines After...,INFORMATIVE,0.626790,0.373210,1,{'testing_quarantine': ['tests']},5,MEDIUM
1,Results from UVRI showed the sample is positiv...,INFORMATIVE,0.909645,0.090355,2,{'case_growth': ['confirmed']},7,MEDIUM
2,"Today or tomorrow, the number of #COVIDー19 cas...",UNINFORMATIVE,0.289060,0.710940,2,{'case_growth': ['cases']},2,LOW
3,Ramsey County veterans experiencing negative f...,UNINFORMATIVE,0.244093,0.755907,0,{},0,LOW
4,The #Covid19 death rate in New Orleans is 7x h...,UNINFORMATIVE,0.357295,0.642705,3,{'death_severity': ['death']},3,LOW


In [17]:
# 위험후보 Top 20 보기
level_rank = {
    "HIGH": 3,
    "MEDIUM": 2,
    "LOW": 1
}

unlabeled_risk_df["alert_rank"] = unlabeled_risk_df["alert_level"].map(level_rank)

top_risk = unlabeled_risk_df.sort_values(
    by=["alert_rank", "final_score", "prob_informative"],
    ascending=[False, False, False]
)

top_risk[
    ["Text", "pred_label", "prob_informative", "keyword_score", "matched_keywords", "final_score", "alert_level"]
].head(20)

,Text,pred_label,prob_informative,keyword_score,matched_keywords,final_score,alert_level
8797,We have 31 new cases of COVID-19 today in Albe...,INFORMATIVE,0.988562,9,"{'case_growth': ['new cases', 'cases'], 'death...",14,HIGH
1213,A man in his 70s who tested positive for COVID...,INFORMATIVE,0.987584,9,"{'case_growth': ['tested positive'], 'death_se...",14,HIGH
2617,"Maybe I missed it, but I hadn't seen official ...",INFORMATIVE,0.961107,9,"{'case_growth': ['cases'], 'death_severity': [...",14,HIGH
7016,We are deeply saddened to note the first death...,INFORMATIVE,0.955529,9,"{'case_growth': ['tested positive'], 'death_se...",14,HIGH
6445,"•03-21 #coronavirus in ITALY 53,578 confirmed ...",INFORMATIVE,0.950349,9,"{'case_growth': ['confirmed cases', 'confirmed...",14,HIGH
9354,31 news cases in Alberta. 226 cases. 16 of the...,INFORMATIVE,0.938778,9,"{'case_growth': ['cases'], 'death_severity': [...",14,HIGH
5258,Update Covid-19 (Italy): 793 deaths today. 19....,INFORMATIVE,0.929274,9,"{'case_growth': ['confirmed', 'cases', 'increa...",14,HIGH
3063,Mark this. Covid-19 (Italy): 793 deaths today....,INFORMATIVE,0.893795,9,"{'case_growth': ['confirmed', 'cases', 'increa...",14,HIGH
5191,US now ranks 4th in confirmed #Covid_19 cases ...,INFORMATIVE,0.803565,9,"{'case_growth': ['confirmed cases', 'confirmed...",14,HIGH
2732,UPDATE: There are now three COVID-19-related d...,INFORMATIVE,0.994145,8,"{'case_growth': ['confirmed'], 'death_severity...",13,HIGH


In [18]:
# CSV 저장
unlabeled_risk_df.to_csv("baseline_with_keyword_risk_results.csv", index=False)
print("saved: baseline_with_keyword_risk_results.csv")

saved: baseline_with_keyword_risk_results.csv


In [19]:
# CSV 내용확인
result_df = pd.read_csv("baseline_with_keyword_risk_results.csv")
result_df.head()

,Text,pred_label,prob_informative,prob_uninformative,keyword_score,matched_keywords,final_score,alert_level,alert_rank
0,Fox Business' Lou Dobbs Self-Quarantines After...,INFORMATIVE,0.626790,0.373210,1,{'testing_quarantine': ['tests']},5,MEDIUM,2
1,Results from UVRI showed the sample is positiv...,INFORMATIVE,0.909645,0.090355,2,{'case_growth': ['confirmed']},7,MEDIUM,2
2,"Today or tomorrow, the number of #COVIDー19 cas...",UNINFORMATIVE,0.289060,0.710940,2,{'case_growth': ['cases']},2,LOW,1
3,Ramsey County veterans experiencing negative f...,UNINFORMATIVE,0.244093,0.755907,0,{},0,LOW,1
4,The #Covid19 death rate in New Orleans is 7x h...,UNINFORMATIVE,0.357295,0.642705,3,{'death_severity': ['death']},3,LOW,1


In [20]:
result_df.columns

Index(['Text', 'pred_label', 'prob_informative', 'prob_uninformative',
       'keyword_score', 'matched_keywords', 'final_score', 'alert_level',
       'alert_rank'],
      dtype='object')

In [21]:
# 전체 행 개수 확인
result_df.shape

(12000, 9)

In [22]:
#alert level 분포 확인
result_df["alert_level"].value_counts()

,count
alert_level,
LOW,6086
MEDIUM,5061
HIGH,853


In [23]:
#정보성 분류 결과 확인
result_df["pred_label"].value_counts()

,count
pred_label,
INFORMATIVE,6949
UNINFORMATIVE,5051
